# Capitolo 4 — Valori mancanti e variabili categoriche: il Titanic (§ 4.8)

In [1]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

def fai_modello(ingressi, nascosti=256, dropout=0.0):
    return nn.Sequential(nn.Linear(ingressi, nascosti), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(nascosti, nascosti), nn.ReLU(), nn.Dropout(dropout), nn.Linear(nascosti, 1))

def addestra(modello, Xtr, ytr, Xva, yva, epoche=300, lr=1e-3, weight_decay=0.0, pazienza=None, pos_weight=None):
    perdita_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight) if pos_weight else None)
    opt = torch.optim.Adam(modello.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr, ytr, Xva, yva = map(torch.from_numpy, (Xtr, ytr, Xva, yva))
    storia = {"train": [], "val": []}; migliore = {"perdita": float("inf"), "pesi": None, "epoca": 0}; attesa = 0
    for epoca in range(epoche):
        modello.train(); opt.zero_grad()
        perdita = perdita_fn(modello(Xtr).squeeze(1), ytr); perdita.backward(); opt.step()
        modello.eval()
        with torch.no_grad(): perdita_val = nn.BCEWithLogitsLoss()(modello(Xva).squeeze(1), yva).item()
        storia["train"].append(perdita.item()); storia["val"].append(perdita_val)
        if perdita_val < migliore["perdita"]:
            migliore = {"perdita": perdita_val, "epoca": epoca, "pesi": {k: v.clone() for k, v in modello.state_dict().items()}}; attesa = 0
        else:
            attesa += 1
            if pazienza is not None and attesa >= pazienza: break
    modello.load_state_dict(migliore["pesi"])
    return storia, migliore

def probabilita(modello, X):
    modello.eval()
    with torch.no_grad(): return torch.sigmoid(modello(torch.from_numpy(X)).squeeze(1)).numpy()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
fissa_seme(42)
df = dati.titanic()
print(df.shape); print(df.isna().sum())

scarico titanic.csv ... 

0.1 MB
(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Mancanti: eliminare, riempire, segnalare

In [2]:
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])       # la cabina manca nel 77% dei casi
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Age_mancante"] = df["Age"].isna().astype(float)

df_tr, df_te = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Survived"])
df_tr, df_te = df_tr.copy(), df_te.copy()
eta_mediana = df_tr["Age"].median()                                     # calcolata sul training
df_tr["Age"] = df_tr["Age"].fillna(eta_mediana); df_te["Age"] = df_te["Age"].fillna(eta_mediana)
print("età mediana usata:", eta_mediana)

età mediana usata: 28.5


## Categoriche: one-hot, con il `reindex` che allinea le colonne

In [3]:
def prepara(d):
    d = d.copy(); d["Sex"] = (d["Sex"] == "female").astype(float)
    return pd.get_dummies(d, columns=["Embarked", "Pclass"], dtype=float)
df_tr, df_te = prepara(df_tr), prepara(df_te)
df_te = df_te.reindex(columns=df_tr.columns, fill_value=0.0)
colonne = [c for c in df_tr.columns if c != "Survived"]; print(colonne)

['Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Age_mancante', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Pclass_1', 'Pclass_2', 'Pclass_3']


In [4]:
X_tr = df_tr[colonne].values.astype(np.float32); y_tr = df_tr["Survived"].values.astype(np.float32)
X_te = df_te[colonne].values.astype(np.float32); y_te = df_te["Survived"].values.astype(np.float32)
X_tr2, X_va, y_tr2, y_va = train_test_split(X_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr)
scaler = StandardScaler().fit(X_tr2)
A, B, C = [scaler.transform(a).astype(np.float32) for a in (X_tr2, X_va, X_te)]

fissa_seme(42)
modello = fai_modello(A.shape[1], nascosti=64, dropout=0.3)
storia, migliore = addestra(modello, A, y_tr2, B, y_va, weight_decay=1e-3, pazienza=40, epoche=500)
pred = (probabilita(modello, C) > 0.5).astype(int)
print(f"accuratezza {accuracy_score(y_te, pred):.1%}  (sempre 'non sopravvive': {1 - y_te.mean():.1%})")
print(f"precisione {precision_score(y_te, pred):.1%}   recall {recall_score(y_te, pred):.1%}   epoca migliore {migliore['epoca'] + 1}")
print(confusion_matrix(y_te, pred))

accuratezza 80.4%  (sempre 'non sopravvive': 61.5%)
precisione 84.0%   recall 60.9%   epoca migliore 120
[[102   8]
 [ 27  42]]
